# Zomato Bangalore EDA

Exploratory data analysis for the local `zomato.csv` file. The notebook loads the data, checks quality issues, cleans key columns, and explores restaurant ratings, costs, locations, cuisines, and service options.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except ModuleNotFoundError:
    HAS_SEABORN = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
if HAS_SEABORN:
    sns.set_theme(style='whitegrid', palette='Set2')
else:
    plt.style.use('ggplot')

DATA_PATH = Path('zomato.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd() / 'zomatio eda' / 'zomato.csv'

DATA_PATH

## 2. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.head()

In [ ]:
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.info(memory_usage='deep')

## 3. Basic Data Quality Checks

In [ ]:
missing_summary = (
    df.isna().sum()
    .to_frame('missing_count')
    .assign(missing_percent=lambda x: (x['missing_count'] / len(df) * 100).round(2))
    .sort_values('missing_percent', ascending=False)
)
missing_summary

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f'Duplicate rows: {duplicate_rows:,}')
df.describe(include='object').T

## 4. Clean Important Columns

In [ ]:
eda = df.copy()

eda.columns = (
    eda.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^0-9a-zA-Z_]', '', regex=True)
)

eda = eda.rename(columns={
    'approx_costfor_two_people': 'cost_for_two',
    'listed_intype': 'listed_type',
    'listed_incity': 'listed_city',
})

eda['rating'] = (
    eda['rate']
    .astype(str)
    .str.extract(r'(\d+(?:\.\d+)?)')[0]
    .astype(float)
)

eda['cost_for_two'] = (
    eda['cost_for_two']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.extract(r'(\d+)')[0]
    .astype(float)
)

for col in ['online_order', 'book_table', 'location', 'rest_type', 'cuisines', 'listed_type', 'listed_city']:
    eda[col] = eda[col].replace('nan', np.nan)

phone_parts = eda['phone'].fillna('').astype(str).str.split(r'\s*\r?\n\s*', n=1, expand=True)
eda['phone1'] = phone_parts[0].replace('', np.nan)
eda['phone2'] = phone_parts[1] if 1 in phone_parts.columns else phone_parts[0]
eda['phone2'] = eda['phone2'].where(eda['phone2'].notna(), phone_parts[0]).replace('', np.nan)

eda[['name', 'phone', 'phone1', 'phone2', 'rating', 'votes', 'cost_for_two', 'online_order', 'book_table', 'location', 'rest_type', 'cuisines']].head()

In [ ]:
clean_summary = eda[['rating', 'votes', 'cost_for_two']].describe().T
clean_summary

## 5. Rating and Cost Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if HAS_SEABORN:
    sns.histplot(eda['rating'].dropna(), bins=25, kde=True, ax=axes[0])
else:
    axes[0].hist(eda['rating'].dropna(), bins=25, edgecolor='white')
axes[0].set_title('Distribution of Restaurant Ratings')
axes[0].set_xlabel('Rating')

if HAS_SEABORN:
    sns.histplot(eda['cost_for_two'].dropna(), bins=40, kde=True, ax=axes[1])
else:
    axes[1].hist(eda['cost_for_two'].dropna(), bins=40, edgecolor='white')
axes[1].set_title('Distribution of Approximate Cost for Two')
axes[1].set_xlabel('Cost for two people')
axes[1].set_xlim(0, eda['cost_for_two'].quantile(0.99))

plt.tight_layout()

## 6. Top Locations and Restaurant Types

In [ ]:
top_locations = eda['location'].value_counts().head(15)
top_rest_types = eda['rest_type'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
if HAS_SEABORN:
    sns.barplot(x=top_locations.values, y=top_locations.index, ax=axes[0])
else:
    axes[0].barh(top_locations.index[::-1], top_locations.values[::-1])
axes[0].set_title('Top Restaurant Locations')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('')

if HAS_SEABORN:
    sns.barplot(x=top_rest_types.values, y=top_rest_types.index, ax=axes[1])
else:
    axes[1].barh(top_rest_types.index[::-1], top_rest_types.values[::-1])
axes[1].set_title('Top Restaurant Types')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('')

plt.tight_layout()

## 7. Online Ordering and Table Booking

In [ ]:
service_summary = (
    eda.groupby(['online_order', 'book_table'], dropna=False)
    .agg(restaurants=('name', 'count'), avg_rating=('rating', 'mean'), avg_cost=('cost_for_two', 'mean'))
    .round(2)
    .sort_values('restaurants', ascending=False)
)
service_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if HAS_SEABORN:
    sns.boxplot(data=eda, x='online_order', y='rating', ax=axes[0])
else:
    online_groups = [group['rating'].dropna() for _, group in eda.groupby('online_order')]
    axes[0].boxplot(online_groups, labels=eda.groupby('online_order').groups.keys())
axes[0].set_title('Rating by Online Order Availability')
axes[0].set_xlabel('Online order')
axes[0].set_ylabel('Rating')

if HAS_SEABORN:
    sns.boxplot(data=eda, x='book_table', y='rating', ax=axes[1])
else:
    booking_groups = [group['rating'].dropna() for _, group in eda.groupby('book_table')]
    axes[1].boxplot(booking_groups, labels=eda.groupby('book_table').groups.keys())
axes[1].set_title('Rating by Table Booking Availability')
axes[1].set_xlabel('Book table')
axes[1].set_ylabel('Rating')

plt.tight_layout()

## 8. Cuisine Analysis

In [ ]:
cuisine_counts = (
    eda['cuisines']
    .dropna()
    .str.split(', ')
    .explode()
    .str.strip()
    .value_counts()
    .head(20)
)

plt.figure(figsize=(10, 7))
if HAS_SEABORN:
    sns.barplot(x=cuisine_counts.values, y=cuisine_counts.index)
else:
    plt.barh(cuisine_counts.index[::-1], cuisine_counts.values[::-1])
plt.title('Top 20 Cuisines')
plt.xlabel('Count')
plt.ylabel('Cuisine')
plt.tight_layout()

## 9. Highly Rated Popular Restaurants

In [ ]:
popular_restaurants = (
    eda.loc[eda['votes'].fillna(0) >= 100]
    .groupby(['name', 'location'], as_index=False)
    .agg(avg_rating=('rating', 'mean'), total_votes=('votes', 'sum'), avg_cost=('cost_for_two', 'mean'))
    .dropna(subset=['avg_rating'])
    .sort_values(['avg_rating', 'total_votes'], ascending=[False, False])
    .head(20)
)

popular_restaurants.round({'avg_rating': 2, 'avg_cost': 0})

## 10. Key Takeaways Template

After running the notebook, summarize your findings here:

- Most common restaurant locations:
- Most common restaurant types:
- Rating pattern by online ordering:
- Rating pattern by table booking:
- Most frequent cuisines:
- Notable highly rated restaurants: